In [1]:
#セル1 --- ライブラリのインポート ---

import matplotlib
matplotlib.use('Agg')  # グラフを画面に出力せずメモリを節約するモード

import tkinter as tk
from tkinter import filedialog
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import re
from scipy.signal import butter, filtfilt
from collections import defaultdict
import openpyxl
from openpyxl.drawing.image import Image as OpenpyxlImage
from openpyxl.utils import get_column_letter
from io import BytesIO

print("セル1: ライブラリのインポートが完了しました。")

セル1: ライブラリのインポートが完了しました。


In [2]:
# セル2--- 設定セクション ---

# ファイル読み込み設定
ROWS_TO_SKIP = 70
FILE_ENCODING = 'cp932'

# 固定の行範囲を指定
START_ROW =  500071      # 抽出開始行
END_ROW =   3500071       # 抽出終了行

# 処理対象の列
TARGET_COLUMNS = ['(1)HA-V05']

# ★ 抽出パラメータの設定 ★
TIME_BEFORE_MAX_SEC = 0.5  # 最高値から何秒前のデータを取得するか
RISE_THRESHOLD_V = 1.9     # 立ち上がりと判定する閾値(V)

# データ処理設定
OFFSET_CALCULATION_ROWS = 1000 
FILTER_CUTOFF_HZ = 500
FILTER_ORDER = 5

print("セル2: 設定の読み込みが完了しました。")

セル2: 設定の読み込みが完了しました。


In [3]:
#セル3 --- 加工関数の定義 ---

def generate_plot_image(df, points_data, original_filepath, output_path, actual_sampling_rate_hz):
    base_filename = os.path.basename(original_filepath)
    fig, axes = plt.subplots(len(TARGET_COLUMNS), 1, figsize=(10, 4 * len(TARGET_COLUMNS)))
    
    if len(TARGET_COLUMNS) == 1: axes = [axes]

    for i, col in enumerate(TARGET_COLUMNS):
        ax = axes[i]
        smooth_col = f'{col}_smooth'
        
        ax.plot(df['Time (s)'], df[col], label='Original', color='lightgray', alpha=0.7)
        ax.plot(df['Time (s)'], df[smooth_col], label='Smoothed', color='blue')

        pts = points_data.get(col, {})
        
        if pd.notna(pts.get('max_time')):
            ax.plot(pts['max_time'], pts['max_val'], 'r*', markersize=12, label='Max Point')
            
        if pd.notna(pts.get('bef_time')):
            ax.plot(pts['bef_time'], pts['bef_val'], 'c*', markersize=12, label=f'{TIME_BEFORE_MAX_SEC}s Before Max')
            
        if pd.notna(pts.get('rise_time')):
            ax.plot(pts['rise_time'], pts['rise_val'], 'g*', markersize=12, label='Rise Point')
            ax.axhline(y=RISE_THRESHOLD_V, color='green', linestyle='--', alpha=0.5, label=f'Threshold ({RISE_THRESHOLD_V}V)')

        # --- ★ 修正：グラフのタイトルにサンプリング周波数（Hz）を追加 ★ ---
        title_text = f"{base_filename} (Sampling: {int(actual_sampling_rate_hz)} Hz)"
        ax.set_title(title_text, fontsize=12, fontweight='bold')
        # -------------------------------------------------------------------
        
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Voltage (V)')
        ax.legend(fontsize='small', loc='upper right')
        ax.grid(True)

    fig.tight_layout()
    
    img_buffer = BytesIO()
    fig.savefig(img_buffer, format='png', dpi=150)
    img_buffer.seek(0)
    
    fig.clear()
    plt.close(fig)
    return img_buffer


def process_file(filepath, make_graph=False):
    actual_sampling_rate_hz = 100000.0
    found_sampling_rate = False        
    
    try:
        with open(filepath, 'r', encoding=FILE_ENCODING, errors='ignore') as f:
            for _ in range(50):
                line = f.readline()
                if not line: break
                if 'サンプリング周期' in line:
                    match = re.search(r'(\d+)\s*μs', line)
                    if match:
                        us_val = float(match.group(1))
                        if us_val > 0:
                            actual_sampling_rate_hz = 1000000.0 / us_val
                            found_sampling_rate = True
                            break
    except Exception as e:
        print(f" -> [警告] ヘッダー読み取りエラー: {e}")

    try:
        df = pd.read_csv(filepath, skiprows=ROWS_TO_SKIP, encoding=FILE_ENCODING, on_bad_lines='skip', nrows=END_ROW, low_memory=False)
    except Exception as e:
        print(f"ファイル '{os.path.basename(filepath)}' 読み込みエラー: {e}")
        return None, None

    try:
        for col in TARGET_COLUMNS:
            if col not in df.columns:
                print(f" -> [警告] 列 '{col}' が見つかりません。")
                df[col] = 0
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

        for col in TARGET_COLUMNS:
            if len(df) >= OFFSET_CALCULATION_ROWS:
                offset = df[col].iloc[:OFFSET_CALCULATION_ROWS].mean()
                df[col] = df[col] - offset

        # 絶対時間軸の作成
        df['Time (s)'] = np.arange(0, len(df)) * (1.0 / actual_sampling_rate_hz)
        
        # 不要な前半行を切り捨て
        df = df.iloc[START_ROW:END_ROW].reset_index(drop=True)
        
        nyquist = 0.5 * actual_sampling_rate_hz
        b, a = butter(FILTER_ORDER, FILTER_CUTOFF_HZ / nyquist, btype='low', analog=False)
        
        for col in TARGET_COLUMNS:
            df[f'{col}_smooth'] = filtfilt(b, a, df[col])
            
    except Exception as e:
        print(f" -> データ前処理エラー: {e}")
        return None, None
    
    results_for_this_file = {'source_file': os.path.basename(filepath)}
    points_data_for_plot = {}
    
    for col in TARGET_COLUMNS:
        smooth_col = f'{col}_smooth'
        
        max_idx = df[smooth_col].idxmax()
        max_val = df.loc[max_idx, smooth_col]
        max_time = df.loc[max_idx, 'Time (s)']
        
        target_time_before = max_time - TIME_BEFORE_MAX_SEC
        valid_rows = df[df['Time (s)'] <= target_time_before]
        
        if valid_rows.empty:
            val_before = np.nan
            actual_time_before = np.nan 
        else:
            closest_smaller_idx = valid_rows.index[-1]
            val_before = df.loc[closest_smaller_idx, smooth_col]
            actual_time_before = df.loc[closest_smaller_idx, 'Time (s)']
            
        rise_candidates = df[df[smooth_col] >= RISE_THRESHOLD_V]
        if not rise_candidates.empty:
            rise_idx = rise_candidates.index[0]
            rise_time = df.loc[rise_idx, 'Time (s)']
            rise_val = df.loc[rise_idx, smooth_col]
        else:
            rise_time = np.nan
            rise_val = np.nan
            
        results_for_this_file[f'{col}_最高値(V)'] = max_val
        results_for_this_file[f'{col}_最高値の時間(s)'] = max_time
        results_for_this_file[f'{col}_{TIME_BEFORE_MAX_SEC}秒前の値(V)'] = val_before
        results_for_this_file[f'{col}_{TIME_BEFORE_MAX_SEC}秒前の時間(s)'] = actual_time_before
        results_for_this_file[f'{col}_立ち上がり値(V)'] = rise_val
        results_for_this_file[f'{col}_立ち上がり時間(s)'] = rise_time
        
        points_data_for_plot[col] = {
            'max_time': max_time, 'max_val': max_val,
            'bef_time': actual_time_before, 'bef_val': val_before,
            'rise_time': rise_time, 'rise_val': rise_val
        }

    image_buffer = None
    if make_graph:
        # ★ generate_plot_image関数にサンプリング周波数(actual_sampling_rate_hz)を渡すように修正
        image_buffer = generate_plot_image(df, points_data_for_plot, filepath, None, actual_sampling_rate_hz)

    return results_for_this_file, image_buffer

print("セル3: 関数の定義が完了しました。")

セル3: 関数の定義が完了しました。


In [4]:
#セル4---加工対象のメインフォルダの指定---

print("フォルダ選択ダイアログを開いています...（画面の裏側に隠れている場合があります）")

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

folder_path = filedialog.askdirectory(title="解析したいCSVファイルが入っている【メインフォルダ】を選択してください")

if not folder_path:
    print("キャンセルされました。")
elif not os.path.isdir(folder_path):
    print("エラー: パスが見つかりません。")
else:
    print(f"選択されたフォルダ: {folder_path}")
    
    search_pattern = os.path.join(folder_path, '**', '*.csv')
    csv_files = glob.glob(search_pattern, recursive=True)
    csv_files = [f for f in csv_files if not os.path.basename(f).startswith('summary_')]
    
    if not csv_files:
        print("エラー: CSVファイルが見つかりませんでした。")
    else:
        print(f"{len(csv_files)}個のCSVファイルを検出しました。")
        files_by_subdir = defaultdict(list)
        for f in csv_files:
            relative_path = os.path.relpath(os.path.dirname(f), folder_path)
            if relative_path == '.': relative_path = "（メインフォルダ直下）"
            files_by_subdir[relative_path].append(f)
            
        print("セル4: 準備完了です。")

フォルダ選択ダイアログを開いています...（画面の裏側に隠れている場合があります）
選択されたフォルダ: C:/Users/0uh2j/OneDrive/Desktop/実験データ2026/本実験データ/202606_本実験温度データ/csv
1020個のCSVファイルを検出しました。
セル4: 準備完了です。


In [5]:
#セル5---数値データの抽出とCSVファイルへの保存---

if not 'csv_files' in locals() or not csv_files:
    print("エラー: セル4を先に実行してください。")
else:
    print("\nデータの抽出を開始します（グラフ描画なし・高速）...")
    all_results = []
    
    for subdir, files in sorted(files_by_subdir.items()):
        print(f"\n--- サブフォルダ: {subdir} ---")
        for filepath in sorted(files):
            print(f"抽出中: {os.path.basename(filepath)}")
            
            results, _ = process_file(filepath, make_graph=False)
            if results: all_results.append(results)

    if all_results:
        output_summary_path = os.path.join(folder_path, 'summary_all_features.csv')
        summary_df = pd.DataFrame.from_records(all_results)
        
        summary_df = summary_df.sort_values('source_file')
        summary_df.rename(columns={'source_file': '元ファイル'}, inplace=True)

        # --- ★ 選択肢B：全角「０１０」などを半角に自動変換する機能 ★ ---
        base_names = summary_df['元ファイル'].str.replace(r'\.csv$', '', case=False, regex=True)
        parts = base_names.str.split('_', expand=True)
        
        # アンダースコアが少なくてもエラーで止まらない安全設計
        speed_col = parts[2] if 2 in parts.columns else pd.Series(index=summary_df.index, dtype=str)
        thickness_col = parts[3] if 3 in parts.columns else pd.Series(index=summary_df.index, dtype=str)
        
        trans_dict = str.maketrans('０１２３４５６７８９', '0123456789')
        speed_str = speed_col.fillna('').astype(str).str.translate(trans_dict)
        thickness_str = thickness_col.fillna('').astype(str).str.replace('mm', '', case=False).str.translate(trans_dict)
        
        summary_df['射出速度[mm/s]'] = pd.to_numeric(speed_str, errors='coerce')
        summary_df['キャビティ厚み[mm]'] = pd.to_numeric(thickness_str, errors='coerce')
        # ------------------------------------------------------------------
        
        final_column_order = ['元ファイル', 'キャビティ厚み[mm]', '射出速度[mm/s]']
        for col in TARGET_COLUMNS:
            final_column_order.extend([
                f'{col}_最高値(V)', f'{col}_最高値の時間(s)',
                f'{col}_{TIME_BEFORE_MAX_SEC}秒前の値(V)', f'{col}_{TIME_BEFORE_MAX_SEC}秒前の時間(s)',
                f'{col}_立ち上がり値(V)', f'{col}_立ち上がり時間(s)'
            ])
            
        summary_df_final = summary_df[final_column_order]
        summary_df_final.to_csv(output_summary_path, index=False, encoding='utf-8-sig')
        print(f"\n★ CSV抽出完了: '{output_summary_path}'")


データの抽出を開始します（グラフ描画なし・高速）...

--- サブフォルダ: 1.0mm\１９０℃ ---
抽出中: 001_１９０℃_０１０_1.0mm.csv
抽出中: 002_１９０℃_０１０_1.0mm.csv
抽出中: 003_１９０℃_０１０_1.0mm.csv
抽出中: 004_１９０℃_０１０_1.0mm.csv
抽出中: 005_１９０℃_０１０_1.0mm.csv
抽出中: 006_１９０℃_０１０_1.0mm.csv
抽出中: 007_１９０℃_０１０_1.0mm.csv
抽出中: 008_１９０℃_０１０_1.0mm.csv
抽出中: 009_１９０℃_０１０_1.0mm.csv
抽出中: 010_１９０℃_０１０_1.0mm.csv
抽出中: 011_１９０℃_０２０_1.0mm.csv
抽出中: 012_１９０℃_０２０_1.0mm.csv
抽出中: 013_１９０℃_０２０_1.0mm.csv
抽出中: 014_１９０℃_０２０_1.0mm.csv
抽出中: 015_１９０℃_０２０_1.0mm.csv
抽出中: 016_１９０℃_０２０_1.0mm.csv
抽出中: 017_１９０℃_０２０_1.0mm.csv
抽出中: 018_１９０℃_０２０_1.0mm.csv
抽出中: 019_１９０℃_０２０_1.0mm.csv
抽出中: 020_１９０℃_０２０_1.0mm.csv
抽出中: 021_１９０℃_０４０_1.0mm.csv
抽出中: 022_１９０℃_０４０_1.0mm.csv
抽出中: 023_１９０℃_０４０_1.0mm.csv
抽出中: 024_１９０℃_０４０_1.0mm.csv
抽出中: 025_１９０℃_０４０_1.0mm.csv
抽出中: 026_１９０℃_０４０_1.0mm.csv
抽出中: 027_１９０℃_０４０_1.0mm.csv
抽出中: 028_１９０℃_０４０_1.0mm.csv
抽出中: 029_１９０℃_０４０_1.0mm.csv
抽出中: 030_１９０℃_０４０_1.0mm.csv
抽出中: 031_１９０℃_０８０_1.0mm.csv
抽出中: 032_１９０℃_０８０_1.0mm.csv
抽出中: 033_１９０℃_０８０_1.0mm.csv
抽出中: 034_１９０℃_０８０_1

In [6]:
#セル6---（任意）グラフ画像の生成と、エクセルファイルへの一覧表示---

if not 'csv_files' in locals() or not csv_files:
    print("エラー: セル4を先に実行してください。")
else:
    print("\nグラフの生成とExcelへの貼り付けを開始します（少し時間がかかります）...")
    all_images_by_subdir = defaultdict(list)
    
    for subdir, files in sorted(files_by_subdir.items()):
        print(f"\n--- サブフォルダ: {subdir} のグラフ生成 ---")
        for filepath in sorted(files):
            print(f"グラフ作成中: {os.path.basename(filepath)}")
            
            _, image_buffer = process_file(filepath, make_graph=True)
            if image_buffer:
                all_images_by_subdir[subdir].append(image_buffer)
                
    if not all_images_by_subdir:
        print("グラフ画像を生成できませんでした。")
    else:
        print("\nすべてのグラフを1つのExcelファイルにまとめています...")
        output_excel_path = os.path.join(folder_path, 'summary_plots_COMBINED.xlsx')
        
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "All_Graphs"
        ws.column_dimensions['A'].width = 30 
        
        current_row = 1
        max_graphs_in_a_row_overall = 0
        
        for subdir, images in sorted(all_images_by_subdir.items()):
            ws.cell(row=current_row, column=1, value=subdir).font = openpyxl.styles.Font(bold=True, size=14)
            current_col_for_images = 2
            max_height_in_row = 0
            
            if len(images) > max_graphs_in_a_row_overall:
                max_graphs_in_a_row_overall = len(images)

            for image_buffer in images:
                image_buffer.seek(0)
                img = OpenpyxlImage(image_buffer)
                if img.height > max_height_in_row:
                    max_height_in_row = img.height
                
                col_letter = get_column_letter(current_col_for_images)
                ws.add_image(img, f'{col_letter}{current_row}')
                current_col_for_images += 1
            
            # --- ★ 修正：縦の余白を追加して行間を広げる ★ ---
            if max_height_in_row > 0:
                ws.row_dimensions[current_row].height = (max_height_in_row * 0.75) + 30  # 下に30ポイントの余白
            
            # サブフォルダが変わるごとに「2行」進める（空白行を挟む）
            current_row += 2 
            # -------------------------------------------------
            
        # --- ★ 修正：横の列幅を広げてグラフ間の間隔を広げる ★ ---
        if max_graphs_in_a_row_overall > 0:
            for i in range(2, max_graphs_in_a_row_overall + 2):
                ws.column_dimensions[get_column_letter(i)].width = 145  # (元は125) 数値を大きくして横間隔を拡張
        # ----------------------------------------------------------
                
        wb.save(output_excel_path)
        print(f"\n★ セル6完了: 全グラフのExcelファイルの保存が完了しました！ -> {os.path.basename(output_excel_path)}")


グラフの生成とExcelへの貼り付けを開始します（少し時間がかかります）...

--- サブフォルダ: 1.0mm\１９０℃ のグラフ生成 ---
グラフ作成中: 001_１９０℃_０１０_1.0mm.csv


C:\Users\0uh2j\AppData\Local\Temp\ipykernel_3856\2393311192.py:38: UserWarning: Glyph 65297 (\N{FULLWIDTH DIGIT ONE}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\0uh2j\AppData\Local\Temp\ipykernel_3856\2393311192.py:38: UserWarning: Glyph 65305 (\N{FULLWIDTH DIGIT NINE}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\0uh2j\AppData\Local\Temp\ipykernel_3856\2393311192.py:38: UserWarning: Glyph 65296 (\N{FULLWIDTH DIGIT ZERO}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\0uh2j\AppData\Local\Temp\ipykernel_3856\2393311192.py:41: UserWarning: Glyph 65297 (\N{FULLWIDTH DIGIT ONE}) missing from font(s) DejaVu Sans.
  fig.savefig(img_buffer, format='png', dpi=150)
C:\Users\0uh2j\AppData\Local\Temp\ipykernel_3856\2393311192.py:41: UserWarning: Glyph 65305 (\N{FULLWIDTH DIGIT NINE}) missing from font(s) DejaVu Sans.
  fig.savefig(img_buffer, format='png', dpi=150)
C:\Users\0uh2j\AppData\Local\Temp\ipykernel_3856\2393311192.py:41: UserWarni

グラフ作成中: 002_１９０℃_０１０_1.0mm.csv
グラフ作成中: 003_１９０℃_０１０_1.0mm.csv
グラフ作成中: 004_１９０℃_０１０_1.0mm.csv
グラフ作成中: 005_１９０℃_０１０_1.0mm.csv
グラフ作成中: 006_１９０℃_０１０_1.0mm.csv
グラフ作成中: 007_１９０℃_０１０_1.0mm.csv
グラフ作成中: 008_１９０℃_０１０_1.0mm.csv
グラフ作成中: 009_１９０℃_０１０_1.0mm.csv
グラフ作成中: 010_１９０℃_０１０_1.0mm.csv
グラフ作成中: 011_１９０℃_０２０_1.0mm.csv


C:\Users\0uh2j\AppData\Local\Temp\ipykernel_3856\2393311192.py:38: UserWarning: Glyph 65298 (\N{FULLWIDTH DIGIT TWO}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\0uh2j\AppData\Local\Temp\ipykernel_3856\2393311192.py:41: UserWarning: Glyph 65298 (\N{FULLWIDTH DIGIT TWO}) missing from font(s) DejaVu Sans.
  fig.savefig(img_buffer, format='png', dpi=150)


グラフ作成中: 012_１９０℃_０２０_1.0mm.csv
グラフ作成中: 013_１９０℃_０２０_1.0mm.csv
グラフ作成中: 014_１９０℃_０２０_1.0mm.csv
グラフ作成中: 015_１９０℃_０２０_1.0mm.csv
グラフ作成中: 016_１９０℃_０２０_1.0mm.csv
グラフ作成中: 017_１９０℃_０２０_1.0mm.csv
グラフ作成中: 018_１９０℃_０２０_1.0mm.csv
グラフ作成中: 019_１９０℃_０２０_1.0mm.csv
グラフ作成中: 020_１９０℃_０２０_1.0mm.csv
グラフ作成中: 021_１９０℃_０４０_1.0mm.csv


...

グラフ作成中: 052_１９０℃_１６０_1.0mm.csv
グラフ作成中: 053_１９０℃_１６０_1.0mm.csv
グラフ作成中: 054_１９０℃_１６０_1.0mm.csv
グラフ作成中: 055_１９０℃_１６０_1.0mm.csv
グラフ作成中: 056_１９０℃_１６０_1.0mm.csv
グラフ作成中: 057_１９０℃_１６０_1.0mm.csv
グラフ作成中: 058_１９０℃_１６０_1.0mm.csv
グラフ作成中: 059_１９０℃_１６０_1.0mm.csv
グラフ作成中: 060_１９０℃_１６０_1.0mm.csv
グラフ作成中: 061_１９０℃_２００_1.0mm.csv
グラフ作成中: 062_１９０℃_２００_1.0mm.csv
グラフ作成中: 063_１９０℃_２００_1.0mm.csv
グラフ作成中: 064_１９０℃_２００_1.0mm.csv
グラフ作成中: 065_１９０℃_２００_1.0mm.csv
グラフ作成中: 066_１９０℃_２００_1.0mm.csv
グラフ作成中: 067_１９０℃_２００_1.0mm.csv
グラフ作成中: 068_１９０℃_２００_1.0mm.csv
グラフ作成中: 069_１９０℃_２００_1.0mm.csv
グラフ作成中: 070_１９０℃_２００_1.0mm.csv
グラフ作成中: 071_１９０℃_２４０_1.0mm.csv
グラフ作成中: 072_１９０℃_２４０_1.0mm.csv
グラフ作成中: 073_１９０℃_２４０_1.0mm.csv
グラフ作成中: 074_１９０℃_２４０_1.0mm.csv
グラフ作成中: 075_１９０℃_２４０_1.0mm.csv
グラフ作成中: 076_１９０℃_２４０_1.0mm.csv
グラフ作成中: 077_１９０℃_２４０_1.0mm.csv
グラフ作成中: 078_１９０℃_２４０_1.0mm.csv
グラフ作成中: 079_１９０℃_２４０_1.0mm.csv
グラフ作成中: 080_１９０℃_２４０_1.0mm.csv
グラフ作成中: 081_１９０℃_２８０_1.0mm.csv
グラフ作成中: 082_１９０℃_２８０_1.0mm.csv
グラフ作成中: 083_１９０℃_２８０_1.0mm.csv
グラフ作成中: 

C:\Users\0uh2j\AppData\Local\Temp\ipykernel_3856\2393311192.py:38: UserWarning: Glyph 65299 (\N{FULLWIDTH DIGIT THREE}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\0uh2j\AppData\Local\Temp\ipykernel_3856\2393311192.py:41: UserWarning: Glyph 65299 (\N{FULLWIDTH DIGIT THREE}) missing from font(s) DejaVu Sans.
  fig.savefig(img_buffer, format='png', dpi=150)


グラフ作成中: 092_１９０℃_３２０_1.0mm.csv
グラフ作成中: 093_１９０℃_３２０_1.0mm.csv
グラフ作成中: 094_１９０℃_３２０_1.0mm.csv
グラフ作成中: 095_１９０℃_３２０_1.0mm.csv
グラフ作成中: 096_１９０℃_３２０_1.0mm.csv
グラフ作成中: 097_１９０℃_３２０_1.0mm.csv
グラフ作成中: 098_１９０℃_３２０_1.0mm.csv
グラフ作成中: 099_１９０℃_３２０_1.0mm.csv
グラフ作成中: 100_１９０℃_３２０_1.0mm.csv

--- サブフォルダ: 1.0mm\２１０℃ のグラフ生成 ---
グラフ作成中: 001_２１０℃_０１０_1.0mm.csv
グラフ作成中: 002_２１０℃_０１０_1.0mm.csv
グラフ作成中: 003_２１０℃_０１０_1.0mm.csv
グラフ作成中: 004_２１０℃_０１０_1.0mm.csv
グラフ作成中: 005_２１０℃_０１０_1.0mm.csv
グラフ作成中: 006_２１０℃_０１０_1.0mm.csv
グラフ作成中: 007_２１０℃_０１０_1.0mm.csv
グラフ作成中: 008_２１０℃_０１０_1.0mm.csv
グラフ作成中: 009_２１０℃_０１０_1.0mm.csv
グラフ作成中: 010_２１０℃_０１０_1.0mm.csv
グラフ作成中: 011_２１０℃_０２０_1.0mm.csv
グラフ作成中: 012_２１０℃_０２０_1.0mm.csv
グラフ作成中: 013_２１０℃_０２０_1.0mm.csv
グラフ作成中: 014_２１０℃_０２０_1.0mm.csv
グラフ作成中: 015_２１０℃_０２０_1.0mm.csv
グラフ作成中: 016_２１０℃_０２０_1.0mm.csv
グラフ作成中: 017_２１０℃_０２０_1.0mm.csv
グラフ作成中: 018_２１０℃_０２０_1.0mm.csv
グラフ作成中: 019_２１０℃_０２０_1.0mm.csv
グラフ作成中: 020_２１０℃_０２０_1.0mm.csv
グラフ作成中: 021_２１０℃_０４０_1.0mm.csv
グラフ作成中: 022_２１０℃_０４０_1.0mm.csv
グラフ作

C:\Users\0uh2j\AppData\Local\Temp\ipykernel_3856\2393311192.py:38: UserWarning: Glyph 65301 (\N{FULLWIDTH DIGIT FIVE}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\0uh2j\AppData\Local\Temp\ipykernel_3856\2393311192.py:41: UserWarning: Glyph 65301 (\N{FULLWIDTH DIGIT FIVE}) missing from font(s) DejaVu Sans.
  fig.savefig(img_buffer, format='png', dpi=150)


グラフ作成中: 012_１９０℃_００５_1.5mm.csv
グラフ作成中: 013_１９０℃_００５_1.5mm.csv
グラフ作成中: 014_１９０℃_００５_1.5mm.csv
グラフ作成中: 015_１９０℃_００５_1.5mm.csv
グラフ作成中: 016_１９０℃_００５_1.5mm.csv
グラフ作成中: 017_１９０℃_００５_1.5mm.csv
グラフ作成中: 018_１９０℃_００５_1.5mm.csv
グラフ作成中: 019_１９０℃_００５_1.5mm.csv
グラフ作成中: 020_１９０℃_００５_1.5mm.csv
グラフ作成中: 021_１９０℃_０１０_1.5mm.csv
グラフ作成中: 022_１９０℃_０１０_1.5mm.csv
グラフ作成中: 023_１９０℃_０１０_1.5mm.csv
グラフ作成中: 024_１９０℃_０１０_1.5mm.csv
グラフ作成中: 025_１９０℃_０１０_1.5mm.csv
グラフ作成中: 026_１９０℃_０１０_1.5mm.csv
グラフ作成中: 027_１９０℃_０１０_1.5mm.csv
グラフ作成中: 028_１９０℃_０１０_1.5mm.csv
グラフ作成中: 029_１９０℃_０１０_1.5mm.csv
グラフ作成中: 030_１９０℃_０１０_1.5mm.csv
グラフ作成中: 031_１９０℃_０２０_1.5mm.csv
グラフ作成中: 032_１９０℃_０２０_1.5mm.csv
グラフ作成中: 033_１９０℃_０２０_1.5mm.csv
グラフ作成中: 034_１９０℃_０２０_1.5mm.csv
グラフ作成中: 035_１９０℃_０２０_1.5mm.csv
グラフ作成中: 036_１９０℃_０２０_1.5mm.csv
グラフ作成中: 037_１９０℃_０２０_1.5mm.csv
グラフ作成中: 038_１９０℃_０２０_1.5mm.csv
グラフ作成中: 039_１９０℃_０２０_1.5mm.csv
グラフ作成中: 040_１９０℃_０２０_1.5mm.csv
グラフ作成中: 041_１９０℃_０４０_1.5mm.csv
グラフ作成中: 042_１９０℃_０４０_1.5mm.csv
グラフ作成中: 043_１９０℃_０４０_1.5mm.csv
グラフ作成中: 